In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

In [2]:
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [3]:
spark = (
    SparkSession.builder
    .appName("PySpark_Tips_Ejercicios")
    .getOrCreate()
)

In [4]:
tips_pd = sns.load_dataset("tips")
tips_pd.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
df = spark.createDataFrame(tips_pd)
df.printSchema()
df.show(5)

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: long (nullable = true)

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
+----------+----+------+------+---+------+----+
only showing top 5 rows


# Ejercicio 1: Crear nuevas columnas con withColumn

## Objetivo
Practicar transformaciones básicas con `withColumn`.

## Instrucciones
A partir del dataset `tips`, realiza lo siguiente:

1. Crear una nueva columna llamada `tip_percentage` que calcule el porcentaje de propina:
   
   tip / total_bill * 100

2. Crear una columna llamada `total_per_person` que calcule cuánto pagó en promedio cada persona:

   total_bill / size

3. Mostrar las columnas:
   - total_bill
   - tip
   - size
   - tip_percentage
   - total_per_person

## Pregunta
¿Qué mesas parecen dejar mayor porcentaje de propina?

In [14]:
df.withColumn("tip_percentage", F.col("tip") / F.col("total_bill") * 100).show(5)
df.withColumn('total_per_person', F.col('total_bill')/F.col('size')).show()
df.withColumn('tip_percentage', F.col('tip')/F.col('total_bill')*100).withColumn('total_per_person', F.col('total_bill')/F.col('size')).select('total_bill', 'tip', 'size', 'tip_percentage', 'total_per_person').show()

+----------+----+------+------+---+------+----+------------------+
|total_bill| tip|   sex|smoker|day|  time|size|    tip_percentage|
+----------+----+------+------+---+------+----+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|14.680764538430255|
+----------+----+------+------+---+------+----+------------------+
only showing top 5 rows
+----------+----+------+------+---+------+----+------------------+
|total_bill| tip|   sex|smoker|day|  time|size|  total_per_person|
+----------+----+------+------+---+------+----+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|             8.495|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|3.4466666666666668|
|     21.01| 3.5|  Male|    No|Sun|Din

# Ejercicio 2: Filtrar registros

## Objetivo
Practicar filtros sobre un DataFrame de PySpark.

## Instrucciones
Filtra el dataset para mostrar solo las cuentas que cumplan estas condiciones:

1. `total_bill` mayor a 20
2. `tip` mayor a 3
3. `time` sea igual a `Dinner`

Luego muestra estas columnas:

- total_bill
- tip
- sex
- smoker
- day
- time

## Pregunta
¿Qué características tienen las cuentas más altas con buenas propinas?

In [23]:
df.filter(df['total_bill']>20).show()
df.filter(df['tip']>3).show()
df.filter(df['time']=='Dinner').show()
df.filter(df['total_bill']>20).filter(df['tip']>3).filter(df['time']=='Dinner').select('total_bill', 'tip', 'sex', 'smoker', 'day', 'time').show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     20.65|3.35|  Male|    No|Sat|Dinner|   3|
|     20.29|2.75|Female|    No|Sat|Dinner|   2|
|     39.42|7.58|  Male|    No|Sat|Dinner|   4|
|      21.7| 4.3|  Male|    No|Sat|Dinner|   2|
|     20.69|2.45|Female|    No|Sat|Dinner|   4|
|     24.06| 3.6|  Male|    No|Sat|Dinner|   3|
|     31.27| 5.0|  Male|    No|Sat|Dinner|   3|
|      30.4| 5.6|  Male|    No|Sun|Dinner|   4|
|     22.23| 5.0|  Male|    No|Sun|Dinner|   2|
|      32.4| 6.0|  Male|    No|Sun|Dinner|   4|
|     28.55|2.05|  Male|    No|Sun|Dinne

# Ejercicio 3: Agrupación con groupBy

## Objetivo
Usar `groupBy` y funciones de agregación.

## Instrucciones
Agrupa el dataset por `day` y calcula:

1. Total de cuentas por día
2. Promedio de `total_bill`
3. Promedio de `tip`
4. Máxima cuenta (`max(total_bill)`)

Ordena el resultado por el promedio de cuenta de mayor a menor.

## Pregunta
¿Qué día tiene el mayor promedio de consumo?

In [28]:
df.groupBy('day').agg(F.count("*").alias('Total Cuentas'),
                      F.avg("total_bill").alias('Promedio, Total Bill'),
                      F.avg("tip").alias('Promedio, tip'),
                      F.max("total_bill").alias('Max, Total Bill')).orderBy(F.col('Promedio, Total Bill').desc()).show()


+----+-------------+--------------------+------------------+---------------+
| day|Total Cuentas|Promedio, Total Bill|     Promedio, tip|Max, Total Bill|
+----+-------------+--------------------+------------------+---------------+
| Sun|           76|  21.409999999999997|3.2551315789473687|          48.17|
| Sat|           87|   20.44137931034483|2.9931034482758623|          50.81|
|Thur|           62|  17.682741935483868| 2.771451612903226|          43.11|
| Fri|           19|   17.15157894736842| 2.734736842105263|          40.17|
+----+-------------+--------------------+------------------+---------------+



# Ejercicio 4: Crear un UDF personalizado

## Objetivo
Aplicar una función personalizada sobre una columna del DataFrame.

## Instrucciones
Crea una función que clasifique el nivel de propina según el valor de `tip`:

- `Baja` si tip < 2
- `Media` si tip >= 2 y tip < 4
- `Alta` si tip >= 4

Luego:

1. Convierte esa función en un UDF
2. Crea una nueva columna llamada `tip_level`
3. Muestra:
   - total_bill
   - tip
   - tip_level

## Pregunta
¿Qué nivel de propina aparece con mayor frecuencia?

In [30]:
def definer(item):
  if item < 2:
    return 'Baja'
  elif item >= 2 and item < 4:
    return 'Media'
  else:
    return 'Alta'



In [32]:
definer_udf = F.udf(definer, StringType())

In [33]:
df.withColumn('tip_level', definer_udf('tip')).select('total_bill', 'tip', 'tip_level').show()

+----------+----+---------+
|total_bill| tip|tip_level|
+----------+----+---------+
|     16.99|1.01|     Baja|
|     10.34|1.66|     Baja|
|     21.01| 3.5|    Media|
|     23.68|3.31|    Media|
|     24.59|3.61|    Media|
|     25.29|4.71|     Alta|
|      8.77| 2.0|    Media|
|     26.88|3.12|    Media|
|     15.04|1.96|     Baja|
|     14.78|3.23|    Media|
|     10.27|1.71|     Baja|
|     35.26| 5.0|     Alta|
|     15.42|1.57|     Baja|
|     18.43| 3.0|    Media|
|     14.83|3.02|    Media|
|     21.58|3.92|    Media|
|     10.33|1.67|     Baja|
|     16.29|3.71|    Media|
|     16.97| 3.5|    Media|
|     20.65|3.35|    Media|
+----------+----+---------+
only showing top 20 rows


# Ejercicio 5: Visualización con Seaborn

## Objetivo
Visualizar los datos del dataset `tips` para identificar patrones de consumo y propinas.

## Instrucciones
Usa el dataset `tips` y realiza lo siguiente:

1. Convierte el DataFrame de PySpark a Pandas.
2. Crea un gráfico de tipo **boxplot**:
   - eje X: `day`
   - eje Y: `total_bill`
3. Crea un gráfico de tipo **scatterplot**:
   - eje X: `total_bill`
   - eje Y: `tip`
   - color (`hue`): `smoker`
4. Crea un gráfico de tipo **histplot**:
   - variable: `total_bill`
   - con curva KDE activada

## Preguntas de análisis
- ¿Qué día tiene cuentas más altas?
- ¿Existe relación entre total de cuenta y propina?
- ¿Cómo se distribuyen las cuentas?